### PTPC_FP8(Per-Token, Per-Channel)

In [8]:
import torch

B, L, H = 1, 512, 4096
FP8_MAX = 448.0
SIMULATED_DTYPE = torch.float32
torch.manual_seed(0)

x = torch.randn(B, L, H).clamp(-10, 10)
w = torch.randn(H, H).clamp(-10, 10)

def quantize_fp8(tensor: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return torch.clamp(tensor / scale, min=-FP8_MAX, max=FP8_MAX).to(torch.float8_e4m3fn)

def sqnr(ref: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    return 20 * torch.log10(torch.norm(ref, p=2) / torch.norm(ref - out, p=2))

x_scale = torch.amax(torch.abs(x), dim=-1, keepdim=True) / FP8_MAX
x_fp8 = quantize_fp8(x, x_scale)
x_deq = x_fp8.to(SIMULATED_DTYPE) * x_scale

print(f"Activation QDQ SQNR : {sqnr(x, x_deq):.2f} dB")

w_scale = torch.amax(torch.abs(w), dim=1, keepdim=True) / FP8_MAX
w_fp8 = quantize_fp8(w, w_scale)
w_deq = w_fp8.to(SIMULATED_DTYPE) * w_scale

print(f"Weight QDQ SQNR : {sqnr(w, w_deq):.2f} dB")

ref = x.squeeze(0) @ w.T
out = (x_fp8.to(SIMULATED_DTYPE).squeeze(0) @ w_fp8.to(SIMULATED_DTYPE).T) * x_scale.squeeze(0) * w_scale.T

print(f"MatMul SQNR : {sqnr(ref, out):.2f} dB")

Activation QDQ SQNR : 31.55 dB
Weight QDQ SQNR : 31.56 dB
MatMul SQNR : 28.53 dB
